# Appendix A — Data Collection and Preparation

**Companion to:** *Weather Conditions Associated with High PM2.5 Concentrations in Sofia: A Mathematical Analysis*.

In this appendix, I document how I obtained and prepared the daily dataset used in the main analysis. I retrieved sensor observations from OpenAQ, obtained hourly wind speed and surface pressure from Open-Meteo, converted both sources to daily observations, merged them by date, and exported the resulting CSV. The workflow follows my original working notebooks, `data_extract.ipynb` and `wind_info.ipynb`.

The dataset used in the submitted analysis covers **2 January 2025 to 4 November 2026** (595 daily rows). The main notebook selects 419 days from this dataset. I use the documented historical end date to keep this data-collection appendix aligned with the analysis; API results may change over time.

**Security:** The OpenAQ key is supplied privately at runtime. Do not publish an older working notebook containing an exposed API key.

## A.1. Environment

I use Python, `requests` and `pandas` in Jupyter. Install the dependencies from the project's `requirements.txt` before running this notebook. The workflow generates its CSV directly from the two APIs; **no pre-existing combined CSV is required**.

## A.2. Data sources and settings

For air quality, I use **OpenAQ API v3**, location **3287527** in Sofia, and retrieve daily observations for `pm25`, `pm10`, `temperature` and `relativehumidity`. My original extraction used a request window from 1 January 2025 to 5 November 2026. I retain this request window but select the observations through **4 November 2026**, the end date of the dataset used in the main analysis.

For weather, I use the **Open-Meteo Historical Weather API** at **42.7066° N, 23.3025° E**, with `timezone=Europe/Sofia`. I request hourly `wind_speed_10m` and `surface_pressure` for the dates covered by the prepared OpenAQ table. The weather series are gridded historical estimates, not measurements from the OpenAQ sensor itself.

In [1]:
from pathlib import Path
import os
import requests
import pandas as pd

OPENAQ_LOCATION_ID = 3287527
LATITUDE, LONGITUDE = 42.7066, 23.3025
TIMEZONE = "Europe/Sofia"
OPENAQ_START, OPENAQ_END = "2025-01-01", "2026-11-05"
DATASET_END = pd.Timestamp("2026-11-04")
OUTPUT_CSV = Path("all_sofia_air_quality_weather_daily.csv")
API_KEY = os.environ.get("OPENAQ_API_KEY", "").strip()
print("OpenAQ API key configured:", bool(API_KEY))
print("Output CSV:", OUTPUT_CSV)


OpenAQ API key configured: False
Output CSV: all_sofia_air_quality_weather_daily.csv


## A.3. OpenAQ authentication

I read the API key from the `OPENAQ_API_KEY` environment variable, or enter it in a hidden prompt. The key is not printed or included in the exported dataset.

In [2]:
from getpass import getpass
if not API_KEY:
    API_KEY = getpass("Enter your NEW OpenAQ API key (hidden): ").strip()
if not API_KEY:
    raise ValueError("No OpenAQ API key was supplied")
print("OpenAQ key available for this session; value not displayed")


Enter your NEW OpenAQ API key (hidden):  ········


OpenAQ key available for this session; value not displayed


## A.4. Downloading OpenAQ observations

I request the location metadata and sensor list using `/locations/3287527` and `/locations/3287527/sensors`. For each selected sensor, I retrieve the daily records from `/sensors/{sensor_id}/days`, requesting successive pages until all records are downloaded. I retain the value, local and UTC timestamps, source-reported unit, and available daily coverage information.

The original dataset includes **PM10**, although the main mathematical analysis focuses on PM2.5. I keep PM10 in the exported table for consistency with the original data-preparation workflow. Sensor units and coverage are displayed for transparency; the original extraction did not exclude low-coverage days.

In [3]:
def get_openaq_daily(api_key, location_id=OPENAQ_LOCATION_ID):
    if not api_key:
        raise RuntimeError("OpenAQ API key is required")
    session = requests.Session()
    session.headers.update({"X-API-Key": api_key})
    base = "https://api.openaq.org/v3"

    def get_json(url, params=None):
        response = session.get(url, params=params, timeout=60)
        response.raise_for_status()
        return response.json()

    location = get_json(f"{base}/locations/{location_id}")
    sensors = []
    page = 1
    while True:
        batch = get_json(f"{base}/locations/{location_id}/sensors",
                         {"limit": 1000, "page": page}).get("results", [])
        sensors.extend(batch)
        if len(batch) < 1000:
            break
        page += 1
    wanted = {"pm25", "pm10", "temperature", "relativehumidity"}
    selected = [s for s in sensors if (s.get("parameter") or {}).get("name") in wanted]
    if not selected:
        raise ValueError("No relevant sensors found at this location")
    sensor_table = pd.DataFrame([{
        "Sensor_ID": s["id"],
        "Pollutant": s["parameter"]["name"],
        "Sensor_Unit": s["parameter"].get("units"),
    } for s in selected]).sort_values(["Pollutant", "Sensor_ID"])
    print("OpenAQ sensors and source-reported units:")
    display(sensor_table)
    if sensor_table["Pollutant"].duplicated().any():
        raise ValueError("Multiple sensors for a parameter: select the historical sensor ID explicitly before merging")
    rows = []
    for sensor in selected:
        sensor_id = sensor["id"]
        parameter = sensor["parameter"]["name"]
        unit = sensor["parameter"].get("units")
        page = 1
        while True:
            batch = get_json(f"{base}/sensors/{sensor_id}/days", {
                "date_from": OPENAQ_START, "date_to": OPENAQ_END,
                "limit": 1000, "page": page}).get("results", [])
            for item in batch:
                dt = ((item.get("period") or {}).get("datetimeFrom") or {})
                daily_unit = (item.get("parameter") or {}).get("units")
                if daily_unit and unit and daily_unit != unit:
                    raise ValueError(f"Unit mismatch for sensor {sensor_id}: {daily_unit} vs {unit}")
                rows.append({
                    "Station_ID": location_id, "Sensor_ID": sensor_id,
                    "Pollutant": parameter, "Value": item.get("value"),
                    "Unit": daily_unit or unit,
                    "Timestamp_Local": dt.get("local"),
                    "Timestamp_UTC": dt.get("utc"),
                    "Coverage": (item.get("coverage") or {}).get("percentComplete"),
                })
            if len(batch) < 1000:
                break
            page += 1
    if not rows:
        raise ValueError("OpenAQ returned no daily observations")
    return pd.DataFrame(rows), location, sensor_table


In [4]:
air_raw, location_metadata, sensor_table = get_openaq_daily(API_KEY)
print("Downloaded OpenAQ records:", len(air_raw))
print("Station name:", (location_metadata.get("results") or [{}])[0].get("name", "not supplied"))
print("Units in retrieved records:")
print(air_raw.groupby(["Sensor_ID", "Pollutant"], dropna=False)["Unit"].unique().to_string())
print("Daily coverage by parameter (%):")
display(air_raw.groupby("Pollutant")["Coverage"].agg(["count", "min", "median", "mean", "max"]).round(2))
print("Rows with reported coverage below 75%:", int(air_raw["Coverage"].lt(75).sum()))
display(air_raw.head(5))


OpenAQ sensors and source-reported units:


,Sensor_ID,Pollutant,Sensor_Unit
2,11440718,pm10,µg/m³
3,11440736,pm25,µg/m³
0,11440699,relativehumidity,%
1,11440705,temperature,c


Downloaded OpenAQ records: 1801
Station name: Sofia, 13 Dimitar Petkov Str.
Units in retrieved records:
Sensor_ID  Pollutant       
11440699   relativehumidity        [%]
11440705   temperature             [c]
11440718   pm10                [µg/m³]
11440736   pm25                [µg/m³]
Daily coverage by parameter (%):


,count,min,median,mean,max
Pollutant,,,,,
pm10,13,4.0,79.0,67.38,100.0
pm25,596,4.0,100.0,98.21,100.0
relativehumidity,596,4.0,100.0,98.21,100.0
temperature,596,4.0,100.0,98.21,100.0


Rows with reported coverage below 75%: 45


,Station_ID,Sensor_ID,Pollutant,Value,Unit,Timestamp_Local,Timestamp_UTC,Coverage
0,3287527,11440699,relativehumidity,46.9,%,2025-01-02T00:00:00+02:00,2025-01-01T22:00:00Z,33.0
1,3287527,11440699,relativehumidity,42.4,%,2025-01-03T00:00:00+02:00,2025-01-02T22:00:00Z,100.0
2,3287527,11440699,relativehumidity,36.4,%,2025-01-04T00:00:00+02:00,2025-01-03T22:00:00Z,100.0
3,3287527,11440699,relativehumidity,34.4,%,2025-01-05T00:00:00+02:00,2025-01-04T22:00:00Z,96.0
4,3287527,11440699,relativehumidity,36.5,%,2025-01-06T00:00:00+02:00,2025-01-05T22:00:00Z,71.0


## A.5. Preparing the daily OpenAQ table

I convert each offset-aware OpenAQ local timestamp to `Europe/Sofia` using `pd.to_datetime(..., utc=True).dt.tz_convert(...)`, then extract the calendar date. This follows the approach I used in my original `data_extract.ipynb`. I also inspect the source's local calendar date and UTC timestamp to identify possible discrepancies in date labels.

I then pivot the observations into one row per date, with separate columns for PM2.5, PM10, temperature and humidity. Before doing so, I check for duplicate date–parameter combinations.

Although I requested OpenAQ observations starting from 1 January 2025, the earliest available daily record was dated 2 January 2025. The original data collection ended on 4 November 2026, so I retained this as the final date rather than including any later observations returned by subsequent API requests. I did not create or estimate values for dates without observations.

Finally, I use the first and last dates of the prepared OpenAQ dataset to define the corresponding historical weather-data request from Open-Meteo. This ensures that both sources cover the same calendar period before I merge them by date.

In [5]:
def inspect_openaq_dates(air_raw):
    air = air_raw.copy()
    air["Date_original_method"] = (pd.to_datetime(air["Timestamp_Local"], utc=True, errors="raise")
        .dt.tz_convert(TIMEZONE).dt.tz_localize(None).dt.normalize())
    air["Date_local_calendar"] = pd.to_datetime(
        air["Timestamp_Local"].astype(str).str.slice(0, 10), errors="raise")
    if air["Timestamp_UTC"].notna().all():
        air["Date_from_utc"] = (pd.to_datetime(air["Timestamp_UTC"], utc=True, errors="raise")
            .dt.tz_convert(TIMEZONE).dt.tz_localize(None).dt.normalize())
    else:
        air["Date_from_utc"] = pd.NaT
    print("Original method vs local-calendar mismatches:",
          int(air["Date_original_method"].ne(air["Date_local_calendar"]).sum()))
    if air["Date_from_utc"].notna().all():
        print("Original method vs UTC-to-Sofia mismatches:",
              int(air["Date_original_method"].ne(air["Date_from_utc"]).sum()))
    return air


def prepare_air_daily(air_raw, date_column="Date_original_method"):
    air = inspect_openaq_dates(air_raw)
    if date_column not in {"Date_original_method", "Date_local_calendar", "Date_from_utc"}:
        raise ValueError("Unknown date mapping")
    if air[date_column].isna().any():
        raise ValueError("Selected date mapping contains missing dates")
    air["Date"] = air[date_column]
    duplicates = air.duplicated(["Date", "Pollutant"], keep=False)
    if duplicates.any():
        display(air.loc[duplicates, ["Sensor_ID", "Pollutant", "Date", "Value"]].head(20))
        raise ValueError("Duplicate date/parameter pairs; investigate sensor selection")
    daily = (air.pivot(index="Date", columns="Pollutant", values="Value")
             .reset_index().rename(columns={"pm25": "PM2,5", "pm10": "PM10",
                                         "temperature": "Temperature",
                                         "relativehumidity": "Humidity"}))
    return daily.sort_values("Date").reset_index(drop=True)


In [6]:
DATE_MAPPING = "Date_original_method"  # Method used in data_extract.ipynb

date_diagnostics = inspect_openaq_dates(air_raw)
display(date_diagnostics[["Timestamp_Local", "Timestamp_UTC", "Date_original_method",
                          "Date_local_calendar", "Date_from_utc"]].head(10))
air_daily = prepare_air_daily(air_raw, DATE_MAPPING)
print("OpenAQ daily dates returned:", len(air_daily))
print("Returned date range:", air_daily["Date"].min(), "to", air_daily["Date"].max())
print("Missing values in OpenAQ daily table:")
print(air_daily.isna().sum())
extra_dates = air_daily.loc[air_daily["Date"].gt(DATASET_END), "Date"]
print("Observations after the dataset end date:",
      extra_dates.dt.strftime("%Y-%m-%d").tolist())
air_selected = air_daily.loc[air_daily["Date"].le(DATASET_END)].copy()
if air_selected.empty:
    raise ValueError("No observations in the selected dataset period")
print("Selected daily OpenAQ rows:", len(air_selected))
display(air_selected.head())


Original method vs local-calendar mismatches: 0
Original method vs UTC-to-Sofia mismatches: 0


,Timestamp_Local,Timestamp_UTC,Date_original_method,Date_local_calendar,Date_from_utc
0,2025-01-02T00:00:00+02:00,2025-01-01T22:00:00Z,2025-01-02,2025-01-02,2025-01-02
1,2025-01-03T00:00:00+02:00,2025-01-02T22:00:00Z,2025-01-03,2025-01-03,2025-01-03
2,2025-01-04T00:00:00+02:00,2025-01-03T22:00:00Z,2025-01-04,2025-01-04,2025-01-04
3,2025-01-05T00:00:00+02:00,2025-01-04T22:00:00Z,2025-01-05,2025-01-05,2025-01-05
4,2025-01-06T00:00:00+02:00,2025-01-05T22:00:00Z,2025-01-06,2025-01-06,2025-01-06
5,2025-01-07T00:00:00+02:00,2025-01-06T22:00:00Z,2025-01-07,2025-01-07,2025-01-07
6,2025-01-08T00:00:00+02:00,2025-01-07T22:00:00Z,2025-01-08,2025-01-08,2025-01-08
7,2025-01-09T00:00:00+02:00,2025-01-08T22:00:00Z,2025-01-09,2025-01-09,2025-01-09
8,2025-01-10T00:00:00+02:00,2025-01-09T22:00:00Z,2025-01-10,2025-01-10,2025-01-10
9,2025-01-11T00:00:00+02:00,2025-01-10T22:00:00Z,2025-01-11,2025-01-11,2025-01-11


Original method vs local-calendar mismatches: 0
Original method vs UTC-to-Sofia mismatches: 0
OpenAQ daily dates returned: 596
Returned date range: 2025-01-02 00:00:00 to 2026-11-05 00:00:00
Missing values in OpenAQ daily table:
Pollutant
Date             0
PM10           583
PM2,5            0
Humidity         0
Temperature      0
dtype: int64
Observations after the dataset end date: ['2026-11-05']
Selected daily OpenAQ rows: 595


Pollutant,Date,PM10,"PM2,5",Humidity,Temperature
0,2025-01-02,113.0,95.1,46.9,7.88
1,2025-01-03,59.8,49.2,42.4,10.40
2,2025-01-04,13.8,12.6,36.4,10.80
3,2025-01-05,62.1,51.5,34.4,10.10
4,2025-01-06,83.0,70.6,36.5,11.30


## A.6. Downloading hourly weather data

Following `wind_info.ipynb`, I determine the Open-Meteo request dates from the minimum and maximum dates of the prepared OpenAQ table. I retrieve hourly `wind_speed_10m` and `surface_pressure` for the same period, using the Sofia time zone. I display the units supplied by the API instead of assuming them from the column names.

In [7]:
def get_openmeteo_hourly(air_daily):
    start_date = pd.to_datetime(air_daily["Date"]).min().strftime("%Y-%m-%d")
    end_date = pd.to_datetime(air_daily["Date"]).max().strftime("%Y-%m-%d")
    print("Original-style Open-Meteo interval:", start_date, "to", end_date)
    response = requests.get("https://archive-api.open-meteo.com/v1/archive", params={
        "latitude": LATITUDE, "longitude": LONGITUDE,
        "start_date": start_date, "end_date": end_date,
        "hourly": ["wind_speed_10m", "surface_pressure"],
        "timezone": TIMEZONE}, timeout=60)
    response.raise_for_status()
    payload = response.json()
    hourly = pd.DataFrame({
        "datetime": payload["hourly"]["time"],
        "WindSpeed": payload["hourly"]["wind_speed_10m"],
        "Pressure": payload["hourly"]["surface_pressure"]})
    return hourly, payload["hourly_units"]


In [8]:
weather_hourly, weather_units = get_openmeteo_hourly(air_selected)
print("Open-Meteo reported units:", weather_units)
print("Hourly weather rows:", len(weather_hourly))
display(weather_hourly.head())

Original-style Open-Meteo interval: 2025-01-02 to 2026-11-04
Open-Meteo reported units: {'time': 'iso8601', 'wind_speed_10m': 'km/h', 'surface_pressure': 'hPa'}
Hourly weather rows: 16128


,datetime,WindSpeed,Pressure
0,2025-01-02T00:00,3.6,961.7
1,2025-01-02T01:00,3.9,961.2
2,2025-01-02T02:00,4.2,960.5
3,2025-01-02T03:00,5.3,960.2
4,2025-01-02T04:00,5.3,959.8


## A.7. Converting hourly weather to daily averages

I convert Open-Meteo's time labels to dates and calculate the arithmetic mean of the available hourly wind-speed and surface-pressure values for each day, as in `wind_info.ipynb`. I also count source time labels and nonmissing hourly values as a data-quality check. Because local time representations around daylight-saving changes can be ambiguous, these counts are diagnostics rather than grounds for automatically discarding dates.

In [9]:
def prepare_weather_daily(weather_hourly):
    weather = weather_hourly.copy()
    weather["datetime"] = pd.to_datetime(weather["datetime"], errors="raise")
    weather["Date"] = weather["datetime"].dt.normalize()
    daily = weather.groupby("Date", as_index=False).agg(
        WindSpeed=("WindSpeed", "mean"), Pressure=("Pressure", "mean"),
        WeatherHours=("datetime", "count"),
        WindHours=("WindSpeed", "count"), PressureHours=("Pressure", "count"))
    local_start = daily["Date"].dt.tz_localize(TIMEZONE)
    local_end = (daily["Date"] + pd.Timedelta(days=1)).dt.tz_localize(TIMEZONE)
    daily["CivilDayElapsedHours"] = ((local_end - local_start) / pd.Timedelta(hours=1)).astype(int)
    daily["SourceHourCountOK"] = (daily["WeatherHours"].eq(24)
                                   & daily["WindHours"].eq(24)
                                   & daily["PressureHours"].eq(24))
    return daily


In [10]:
weather_daily = prepare_weather_daily(weather_hourly)
print("Daily Open-Meteo rows:", len(weather_daily))
print("Days with fewer/more than 24 source time labels:",
      int(weather_daily["WeatherHours"].ne(24).sum()))
print("Days with missing hourly wind/pressure values:",
      int((~weather_daily["SourceHourCountOK"]).sum()))
print("Daylight-saving dates (elapsed civil-day hours differ from 24):")
display(weather_daily.loc[weather_daily["CivilDayElapsedHours"].ne(24),
                          ["Date", "WeatherHours", "CivilDayElapsedHours", "WindHours", "PressureHours"]])


Daily Open-Meteo rows: 672
Days with fewer/more than 24 source time labels: 0
Days with missing hourly wind/pressure values: 0
Daylight-saving dates (elapsed civil-day hours differ from 24):


,Date,WeatherHours,CivilDayElapsedHours,WindHours,PressureHours
87,2025-03-30,24,23,24,24
297,2025-10-26,24,25,24,24
451,2026-03-29,24,23,24,24
661,2026-10-25,24,25,24,24


## A.8. Merging and checking the dataset

I merge the daily weather table into the OpenAQ table using a **left join on `Date`**, retaining the dates supplied by OpenAQ. I check for duplicate dates, report missing values and preserve the original column layout: `Date`, `PM10`, `PM2,5`, `Humidity`, `Temperature`, `WindSpeed`, `Pressure`.

PM10 has substantial missing coverage and is not used in the main PM2.5 analysis. The other five study variables are checked for missing values before export.

In [11]:
def merge_and_validate(air_daily, weather_daily):
    if air_daily["Date"].duplicated().any() or weather_daily["Date"].duplicated().any():
        raise ValueError("Duplicate dates before merge")
    merged = air_daily.merge(weather_daily, on="Date", how="left", validate="one_to_one")
    original_cols = ["Date", "PM10", "PM2,5", "Humidity", "Temperature", "WindSpeed", "Pressure"]
    required = ["Date", "PM2,5", "Humidity", "Temperature", "WindSpeed", "Pressure"]
    absent = set(original_cols) - set(merged.columns)
    if absent:
        raise ValueError(f"Missing expected columns: {sorted(absent)}")
    print("Merged rows:", len(merged))
    print("Merged date range:", merged["Date"].min(), "to", merged["Date"].max())
    print("Missing required study values:")
    print(merged[required].isna().sum())
    print("PM10 missing values:", int(merged["PM10"].isna().sum()))
    if merged[required].isna().any().any():
        raise ValueError("Missing required study values after left merge")
    return merged[original_cols].sort_values("Date").reset_index(drop=True)


In [12]:
combined = merge_and_validate(air_selected, weather_daily)
print("Daily combined dataset prepared.")
display(combined.head())


Merged rows: 595
Merged date range: 2025-01-02 00:00:00 to 2026-11-04 00:00:00
Missing required study values:
Date           0
PM2,5          0
Humidity       0
Temperature    0
WindSpeed      0
Pressure       0
dtype: int64
PM10 missing values: 582
Daily combined dataset prepared.


,Date,PM10,"PM2,5",Humidity,Temperature,WindSpeed,Pressure
0,2025-01-02,113.0,95.1,46.9,7.88,4.675000,958.112500
1,2025-01-03,59.8,49.2,42.4,10.40,5.541667,950.679167
2,2025-01-04,13.8,12.6,36.4,10.80,8.375000,955.729167
3,2025-01-05,62.1,51.5,34.4,10.10,5.079167,955.612500
4,2025-01-06,83.0,70.6,36.5,11.30,3.429167,953.004167


## A.9. Exporting the combined CSV

Finally, I export the daily table as `all_sofia_air_quality_weather_daily.csv`. This file contains the air-quality and weather variables used by the main notebook. I report the file path, dimensions and date range so the output can be checked after execution.

In [13]:
combined.to_csv(OUTPUT_CSV, index=False)
print("Saved CSV:", OUTPUT_CSV.resolve())
print("Rows:", len(combined), "| Columns:", len(combined.columns))
print("Date range:", combined["Date"].min().date(), "to", combined["Date"].max().date())
print("Missing values by column:")
print(combined.isna().sum())


Saved CSV: C:\Users\PC\Downloads\all_sofia_air_quality_weather_daily.csv
Rows: 595 | Columns: 7
Date range: 2025-01-02 to 2026-11-04
Missing values by column:
Date             0
PM10           582
PM2,5            0
Humidity         0
Temperature      0
WindSpeed        0
Pressure         0
dtype: int64


## A.10. Data provenance, limitations and use in the main analysis

This appendix documents the full data-collection sequence: **OpenAQ sensor-day retrieval → daily air-quality table → Open-Meteo hourly retrieval → daily weather averages → date-based merge → CSV export**. The resulting historical dataset contains 595 daily records for the period 2 January 2025–4 November 2026. The main research notebook applies its own date filter and analyzes 419 observations.

The OpenAQ data come from one Sofia monitoring location. Weather values are historical estimates for nearby coordinates, so they are not necessarily identical to conditions at the sensor. Daily averaging can conceal within-day variation; some PM10 values are unavailable; and the original workflow does not exclude observations based on the reported OpenAQ coverage percentage. Since APIs may revise historical data, rerunning the notebook at a later date may produce differences. These limitations do not change the retrieval and preparation steps documented here.

**Working sources:** `data_extract.ipynb` and `wind_info.ipynb` (the original development notebooks). Only a version without embedded credentials should be shared.